# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarahabumandil/FlyRank-ML-Intership-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** one row = one content page belonging to one (anonymized) client. Not a client, not a keyword, not a click — 30,000 pages spread across 32 client accounts.

**Time window:** every metric column is a trailing aggregate, not a dated event. Two windows are stacked on top of each other:
- `*_90d` columns cover a rolling 90-day window ending at export time.
- That 90-day window is split in half: `*_last_30d` (most recent 30 days) vs. `*_prev_30d` (the 30 days before that, days 31–60 back). This last-vs-prev split is exactly what `trend_direction` compares to build the label — which is why it matters so much for Section 2.

There's no calendar date anywhere in the file — only day-counts relative to that export moment (`content_age_days`, `days_since_last_update`). I can say how *wide* the window is and how *old* a page is inside it, but not *when* it happened, so anything seasonal is invisible to me here.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("Shape:", df.shape)

# Grain check: does content_id uniquely identify a row?
dupes = df.groupby("content_id").size()
print("content_id is unique per row:", (dupes > 1).sum() == 0, "| duplicate ids found:", (dupes > 1).sum())
print("Distinct clients:", df["client_id"].nunique())

# Window check: 90d total should be >= last_30d + prev_30d (it also covers days 61-90)
covers = df["impressions_90d"] >= (df["impressions_last_30d"] + df["impressions_prev_30d"])
print("impressions_90d >= last_30d + prev_30d for every row:", covers.all())

print("content_age_days range:", df["content_age_days"].min(), "-", df["content_age_days"].max(), "days")
print("days_since_last_update range:", df["days_since_last_update"].min(), "-", df["days_since_last_update"].max(), "days")


Shape: (30000, 44)
content_id is unique per row: True | duplicate ids found: 0
Distinct clients: 32
impressions_90d >= last_30d + prev_30d for every row: True
content_age_days range: 90 - 564 days
days_since_last_update range: 1 - 373 days


## 2. Fields: feature / label / context / excluded

**Feature** (knowable characteristics of the page, safe to learn from):
`search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`, `word_count`, `char_count`, `content_age_days`, `age_tier`, `days_since_last_update`, `freshness_tier`, `days_with_impressions`, `days_with_sessions`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, and the 90d traffic totals (`impressions_90d`, `clicks_90d`, `sessions_90d`, `ai_sessions_90d`, log-transformed) plus their tier buckets.

**Label / proxy:** `trend_direction` and `trend_pct` — `is_declining_label` is computed directly from `trend_direction == "down"`. Never features, per the Week 1/2 contract.

**Context** (for grouping/joining/splitting, never for the model to learn from): `content_id` (row id), `client_id` (must drive a **client-holdout** split, not a random row split — otherwise the same client leaks across train/test).

**Excluded, with why:**
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` — these are the *exact two numbers* `trend_direction` compares to build the label. Section 3 shows that ranking on their raw gap alone reconstructs the label almost perfectly — that's not signal, that's the label wearing a different hat.
- `provider_used`, `model_used` — 71% and 19% missing respectively, and they describe *how the content was produced* (a production/vendor decision), not a quality or engagement signal an editor would act on. The data dictionary flags both as "not a model feature."
- `pageviews_90d`, `users_90d`, `engaged_sessions_90d`, `scroll_events_90d` (raw counts) — redundant with the rate features I'm already keeping (`engagement_rate`, `scroll_rate`) and the log-transformed totals; keeping both the raw count and its own denominator double-counts the same signal.

**One honest caveat, not a full exclusion:** the 90d totals and rates (`impressions_90d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`) are aggregated over a window that *includes* the last-30d third used to build the label. That's diluted with two more months of history, not a direct restatement of it, and the shipped model's Precision@50 = 0.74 (not 1.0) says it isn't trivially solving the label — but it's still worth flagging as a softer leakage risk if this task is ever reframed as *forecasting future* decline rather than ranking *already-observed* decline.

In [2]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

feature_cols = [
    "search_volume", "competition", "competition_level", "cpc", "content_type", "main_intent",
    "word_count", "char_count", "content_age_days", "age_tier", "days_since_last_update", "freshness_tier",
    "days_with_impressions", "days_with_sessions", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
]
label_cols = ["trend_direction", "trend_pct"]
context_cols = ["content_id", "client_id"]
excluded_cols = [
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "provider_used", "model_used",
    "pageviews_90d", "users_90d", "engaged_sessions_90d", "scroll_events_90d",
]

all_sorted = set(feature_cols + label_cols + context_cols + excluded_cols)
remaining = [c for c in df.columns if c not in all_sorted]
print("Every touched field sorted:", len(all_sorted), "columns")
print("Columns not yet sorted (tier/order helpers, kept alongside their base feature):", remaining)


Every touched field sorted: 39 columns
Columns not yet sorted (tier/order helpers, kept alongside their base feature): ['age_tier_order', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier', 'is_declining_label']


## 3. Verify it with queries (grain, counts, missing values, windows)

**Grain:** confirmed above — 30,000 distinct `content_id`, zero duplicates.

**Counts:** 32 distinct `client_id`, matching the data dictionary.

**Missingness (systematic, not random — checked by category):** `search_volume`/`competition`/`cpc` are missing for 2,468 rows, and that missingness is 100% concentrated in `feedly article` rows (no keyword data ever attached to that content type) — a blind `fillna(0)` would silently teach the model "feedly article." `word_count`/`char_count` are missing for a *different* reason: 7,699 rows, concentrated in `keyword article` rows (28.3% missing there, 0% elsewhere) — two separate missingness patterns tied to two different `content_type` values, not one blanket "data quality" problem.

**The leakage check:** if `impressions_last_30d`/`impressions_prev_30d` really do reconstruct the label, ranking on their raw gap alone (no `trend_direction`, no `trend_pct` touched) should score suspiciously well. It does — confirming the exclusion in Section 2 was correct.

In [3]:
# Missingness by content_type: two different patterns, not one
print("search_volume missing rate by content_type:")
print(df.groupby("content_type")["search_volume"].apply(lambda s: round(s.isna().mean(), 3)))
print()
print("word_count missing rate by content_type:")
print(df.groupby("content_type")["word_count"].apply(lambda s: round(s.isna().mean(), 3)))
print()

# Leakage check: rank purely on the raw last30/prev30 gap, no label columns touched
leak_score = df["impressions_prev_30d"] - df["impressions_last_30d"]
top50_leak = df.assign(leak_score=leak_score).sort_values("leak_score", ascending=False).head(50)
print("Precision@50 from raw last30/prev30 gap alone:", round(top50_leak["is_declining_label"].mean(), 3))
print("(Shipped model, using only the approved feature list, reaches 0.740 — the gap above is suspiciously higher, which is exactly the leakage signature I excluded these columns for.)")


search_volume missing rate by content_type:
content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014
Name: search_volume, dtype: float64

word_count missing rate by content_type:
content_type
comparison article    0.000
feedly article        0.000
keyword article       0.283
Name: word_count, dtype: float64

Precision@50 from raw last30/prev30 gap alone: 1.0
(Shipped model, using only the approved feature list, reaches 0.740 — the gap above is suspiciously higher, which is exactly the leakage signature I excluded these columns for.)


## 4. Data limits

**No calendar time.** Everything is a trailing day-count, so I can never say *when* a page declined, only that it did within this snapshot — no seasonality, no before/after a specific algorithm update, no way to check if all 32 clients' windows even line up in real time.

**Unbalanced, systematic missingness.** `provider_used` is missing for 71% of rows and `model_used` for 19% — too sparse and too structurally tied to production workflow to trust as a signal, which is why both are excluded rather than imputed.

**A quiet zero that isn't zero.** `avg_position` is `0` for 1,205 rows, and per the data dictionary that means "no position data," not literally position zero (which would be an impossibly good rank). Left unhandled, those rows would look like the best-performing pages in the dataset instead of the ones with the least information.

**Only 32 clients.** Any pattern I find is a pattern across 32 accounts' worth of content strategy, not the open web — it won't generalize to a client whose content mix or vertical looks nothing like these 32.

**Observed, not forecast.** As in Week 1/2: this ranks pages that already show a declining pattern within the snapshot window. It's decision-support for a human editor today, not a forecast of what any individual page will do next month.

In [4]:
print("avg_position == 0 rows (no position data, not literal position zero):", (df["avg_position"] == 0).sum())
print("provider_used missing:", df["provider_used"].isna().mean().round(3), "| model_used missing:", df["model_used"].isna().mean().round(3))
print("trend_pct missing (filled with 0 upstream when prev_30d impressions = 0):", df["trend_pct"].isna().sum())


avg_position == 0 rows (no position data, not literal position zero): 1205
provider_used missing: 0.715 | model_used missing: 0.191
trend_pct missing (filled with 0 upstream when prev_30d impressions = 0): 3388


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.